In [1]:
import os
import uuid
from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding, SparseTextEmbedding
from dotenv import load_dotenv

load_dotenv()



/Users/luizfelipew/Documents/git/AI-Engineering/dev-eficiente-IA/engineering-ai/curso-ia/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
DENSE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SPARSE_MODEL = "Qdrant/bm25"
COLLECTION_NAME = "financial"
FILE_PATH = "./AAPL_10-K_1A_temp.md"

qdrant = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
)


In [3]:
qdrant.delete_collection(COLLECTION_NAME)

True

In [4]:

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "dense":models.VectorParams (size=384,distance=models.Distance.COSINE)
    },
    sparse_vectors_config={"sparse": models.SparseVectorParams()}
)

True

In [5]:
from markdown_it.rules_block.paragraph import paragraph
with open(FILE_PATH, "r", encoding="utf-8") as f:
    content = f.read()
    
paragraphs = content.split("\n\n")
chunks = [p.strip() for p in paragraphs if len(p.strip()) > 50]

chunks[0]



'The Company’s business, reputation, results of operations, financial condition and stock price can be affected by a number of factors, whether currently known or unknown, including those described below. When any one or more of these risks materialize from time to time, the Company’s business, reputation, results of operations, financial condition and stock price can be materially and adversely affected.'

In [6]:
dense_model = TextEmbedding(DENSE_MODEL)
sparse_model = SparseTextEmbedding(SPARSE_MODEL)

points = []
for chunk in chunks:
    dense_embedding = list(dense_model.passage_embed([chunk]))[0].tolist()
    sparse_embedding = list(sparse_model.passage_embed([chunk]))[0].as_object()
    
    point = models.PointStruct(
        id=str(uuid.uuid4()),
        vector={
            "dense": dense_embedding,
            "sparse": sparse_embedding,
        },
        payload={"text": chunk, "source": FILE_PATH},
    )
    points.append(point)

qdrant.upload_points(collection_name=COLLECTION_NAME, points=points)


Fetching 18 files: 100%|██████████| 18/18 [00:00<00:00, 28.85it/s]


In [7]:
query_text = "What are the main financial risks?"
query_dense = list(dense_model.query_embed([query_text]))[0].tolist()
query_sparse = list(sparse_model.query_embed([query_text]))[0].as_object()

results = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    prefetch=[
        {"query": query_dense, "using": "dense", "limit": 10},
        {"query": query_sparse, "using": "sparse", "limit": 10},
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=3,
)

In [9]:
for r in results.points:
    print(f"Score: {r.score}")
    print(f"Texto: {r.payload['text'][:100]}...")
    print("-" * 80)

Score: 0.6666667
Texto: The Company’s business, reputation, results of operations, financial condition and stock price can b...
--------------------------------------------------------------------------------
Score: 0.6666667
Texto: Adverse economic conditions can also lead to increased credit and collectibility risk on the Company...
--------------------------------------------------------------------------------
Score: 0.5909091
Texto: The Company’s investments can be negatively affected by changes in liquidity, credit deterioration, ...
--------------------------------------------------------------------------------
